In [8]:
# Задание 1 (синтаксис)

# Напишите функцию check_car_id(car_id: str) → str.
 
#  Формат номера: 1 буква + 3 цифры + 2 буквы + 2–3 цифры региона.
#  Буквы только из набора, допустимого в РФ-номерах: А, В, Е, К, М, Н, О, Р, С, Т, У, Х (кириллица). Учитывайте путаницу с латиницей на входе - нормализуйте или явно опишите правило.
 
#  • Если номер валиден: «Номер А222ВС валиден. Регион: 96.»
#  • Если невалиден: «Номер не валиден.»

import re

def check_car_id(car_id: str) -> str:
    pattern = r"([АВЕКМНОРСТУХABEKMHOPCTX]\d{3}[АВЕКМНОРСТУХABEKMHOPCTX]{2})(\d{2,3})"
    m = re.fullmatch(pattern, car_id)  
    if m:
        return f'Номер {m.group(1)}. Регион {m.group(2)}'
    return 'Номер не валиден'


print(check_car_id('А222BС96'))
print(check_car_id('А123ВЕ777'))
print(check_car_id('АБ22ВВ193'))
print(check_car_id('A222BC96'))
print(check_car_id('А222ВС9') )
print(check_car_id('А222ВС9699'))
print(check_car_id('Я222ВС96'))
print(check_car_id(''))

Номер А222BС. Регион 96
Номер А123ВЕ. Регион 777
Номер не валиден
Номер A222BC. Регион 96
Номер не валиден
Номер не валиден
Номер не валиден
Номер не валиден


In [3]:
# Задание 2 (повышенная сложность)

# Дан многострочный лог. Реализуйте parse_logs(text: str) → list[dict].
 
#  1. Одной (или составной) регуляркой извлеките строки только в формате:
#     YYYY-MM-DD HH:MM:SS LEVEL ...
#  2. Из хвоста строки достаньте user, action, ip; если есть card=... - тоже.
#  3. Валидация через regex:
#     • email (разумный практический паттерн);
#     • IPv4;
#     • карта: маска XXXX-XXXX-XXXX-XXXX; в результате хранить только ****-****-****-1111 (последние 4 цифры).
#  4. Невалидные поля не роняют парсер: в словаре "errors": ["invalid_email", ...].
#  5. Бонус: удалите последовательные повторы слов в текстовом поле сообщения через re.

# Входной пример:

# 2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
#  2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
#  2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
#  Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5

import re
from datetime import datetime

def parse_logs(text: str) -> list[dict]:
    new_text = text.splitlines()
    result = list()
    line_re = re.compile(r'^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+(?P<level>INFO|DEBUG|WARN|ERROR)\s+(?P<end>.*)')
    
    email_re = re.compile(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')
    ip_re    = re.compile(r'^(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)$')
    card_re  = re.compile(r'^\d{4}-\d{4}-\d{4}-\d{4}$')
    
    
    
    
    for line in new_text:
        errors = list()
        line = line.strip()
        # print(line.strip())
        if not line:
            continue
        
        m = line_re.match(line)
        if not m:
            continue
        # print(m.groupdict()) 
        date_str = m.group('ts')
        
        
        try:
            datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S')
        except ValueError:
            continue 
        
        server_str = m.group('level')
        # print(date_str)
        # print(re.findall(r'(\w+)=(\S+)', m.group('end')))
        end_list = dict(re.findall(r'(\w+)=(\S+)', m.group('end')))
        # print(end_list)
        
        
        
        
        user = end_list.get('user')
        
        if user is None:
            errors.append('invalid_user')
        elif not email_re.match(user):
            errors.append('invalid_email')
            
        ip = end_list.get('ip')
        
        if ip is None:
            errors.append('invalid_ip')
        elif not ip_re.match(ip):
            errors.append('invalid_ip')
            
        card_rw = end_list.get('card')
        card = None
        if card_rw is not None:
            if card_re.match(card_rw):
                card = '****-****-****-' + card_rw[-4:]
            else:
                errors.append('invalid_card')
        
        
        
        
        rs = dict()
        rs = {
            'ts': date_str,
            'level': server_str,
            'user': user,
            'action': end_list.get('action'),
            'ip': ip,
            'card': card,
            'errors': errors,
        }
        result.append(rs)
        
        
        
    # new_text = re.split("\n",text)   
     
    # pattern = r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} \w+'    
    
    # for line in new_text.splitlines():
    #     # print(re.fullmatch(pattern, line))
    #     print(line)
            
    return result

with open("logs_hw4.txt", encoding="utf-8") as f:
    fstr = f.read()


print(parse_logs(fstr))


[{'ts': '2024-03-15 10:51:25', 'level': 'WARN', 'user': 'user+tag@gmail.com', 'action': 'login', 'ip': '203.0.113.42', 'card': None, 'errors': []}, {'ts': '2024-03-15 11:46:57', 'level': 'DEBUG', 'user': 'petr.sidorov@yandex.ru', 'action': 'login', 'ip': '8.8.8.8', 'card': None, 'errors': []}, {'ts': '2024-03-15 09:36:54', 'level': 'WARN', 'user': 'petr.sidorov@yandex.ru', 'action': 'retry', 'ip': '999.1.1.1', 'card': None, 'errors': ['invalid_ip']}, {'ts': '2024-03-15 09:07:09', 'level': 'ERROR', 'user': 'user@domain.', 'action': 'delete', 'ip': '0.0.0.0', 'card': None, 'errors': ['invalid_email']}, {'ts': '2024-03-15 10:28:11', 'level': 'ERROR', 'user': '@onlydomain.com', 'action': 'pay', 'ip': '300.300.300.300', 'card': None, 'errors': ['invalid_email', 'invalid_ip', 'invalid_card']}, {'ts': '2024-03-15 10:04:06', 'level': 'ERROR', 'user': 'user+tag@gmail.com', 'action': 'pay', 'ip': '127.0.0.1', 'card': None, 'errors': ['invalid_card']}, {'ts': '2024-03-15 10:59:38', 'level': 'DEBU